In [96]:
import os 
from dotenv import load_dotenv
load_dotenv()
gpk =os.getenv("GPK")
#print(gpk)

from langchain_groq import ChatGroq

In [18]:
#!pip install bs4

In [51]:
os.environ["token"] = os.getenv("token")

from langchain_huggingface import HuggingFaceEmbeddings

emb =HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-V2")

llm = ChatGroq(groq_api_key = gpk , model= "llama-3.1-8b-instant")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001B1390F23C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B139716630>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [22]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

In [38]:
import bs4
loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/"),
    bs_kwargs = dict(
        parse_only=bs4.SoupStrainer(
            class_ = ("post-content","post-title","post-header")
        ),
    )
)
#print(loader)
docs = loader.load()

In [48]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 200 , chunk_overlap=20)
splits = text_splitter.split_documents(docs)
#splits

vs = Chroma.from_documents(documents = splits , embedding=emb )
re  = vs.as_retriever()
re

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001B134D77200>, search_kwargs={})

## Prompt Template 

In [49]:
system_prompt =(
    "your are an assistant for question-answering task."
    "Use the following pieces of retrieved context to answer"
    "the question. If you don't know the answer , say thank you "
    "don't know . Use three sentences maximum and keep you "
    "answer concise"
    "\n\n"
    "{context}"
)

prompt =  ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human", "{input}"),
    ])

In [52]:
question_answer = create_stuff_documents_chain(llm , prompt)
rag_chain=create_retrieval_chain(re, question_answer)

In [55]:
response = rag_chain.invoke({"input":"what is self-Reflection"})
response["answer"]

"Self-reflection is the process of examining and exploring one's own thoughts, feelings, and behaviors to gain a deeper understanding of oneself. It involves taking a step back to analyze one's experiences, successes, failures, and emotions to identify areas for personal growth and improvement. Through self-reflection, individuals can develop greater self-awareness, self-acceptance, and make informed decisions about their lives."

In [56]:
response = rag_chain.invoke({"input":"what is kingdom"})
response["answer"]

"Thank you don't know"

In [59]:
response = rag_chain.invoke({"input":"how to achieve it "})
response["answer"]

"Thank you don't know. I need more context to provide a specific answer."

# Adding the Chat History 

In [64]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder 

contextualize_q_systemt_promts = (
    "Given a chat history and latest user question "
    "Which might reference context in the chat history, "
    "formulate a standalone question which can be understand "
    "without the chat history. Do not answe the question, "
    "just reformulate it if needed and otherwise return it as is.")

contextualize_q_promts =  ChatPromptTemplate.from_messages(
    [
        ("system",contextualize_q_systemt_promts),
        MessagesPlaceholder("chat_history"),
        ("human","{input}"),
    ])

In [65]:
his = create_history_aware_retriever(llm , re, contextualize_q_promts)
his

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001B134D77200>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessag

In [66]:
qa_promts =  ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human","{input}"),
    ])

In [67]:
question_answer = create_stuff_documents_chain(llm , qa_promts)
rag_chain=create_retrieval_chain(his, question_answer)

In [82]:
from langchain_core.messages import AIMessage, HumanMessage

chat_his = []

question = "what is self-Reflection"
response1 = rag_chain.invoke({"input": question, "chat_history": chat_his})
print(response1['answer'])

chat_his.extend([
    HumanMessage(content=question),
    AIMessage(content=response1['answer'])
])

question2 = "How to achieve it"
response2 = rag_chain.invoke({"input": question2, "chat_history": chat_his})
print(response2['answer'])

chat_his.extend([
    HumanMessage(content=question2),
    AIMessage(content=response2['answer'])
])

question3 = "Who developed this "
response3 = rag_chain.invoke({"input": question3, "chat_history": chat_his})
print(response2['answer'])

chat_his.extend([
    HumanMessage(content=question3),
    AIMessage(content=response3['answer'])
])


Self-reflection is the process of examining and exploring one's own thoughts, feelings, and behaviors to gain a deeper understanding of oneself. It involves taking a step back to reflect on one's experiences, actions, and emotions to identify patterns, strengths, and areas for improvement. This helps individuals develop self-awareness, make informed decisions, and achieve personal growth.
To achieve self-reflection, consider the following steps:

1. Set aside dedicated time for introspection.
2. Identify and challenge your thoughts and emotions.
3. Reflect on past experiences and identify patterns.
4. Explore your values, goals, and motivations.
5. Practice journaling, meditation, or talking to a trusted friend or mentor.

Thank you don't know
To achieve self-reflection, consider the following steps:

1. Set aside dedicated time for introspection.
2. Identify and challenge your thoughts and emotions.
3. Reflect on past experiences and identify patterns.
4. Explore your values, goals, a

# Entire Flow 

In [95]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_ses(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(
    rag_chain,
    get_ses,
    input_messages_key="input",       # "input": "your user message"
    history_messages_key="chat_history",  # will be injected automatically
    output_messages_key="answer"      # returned by rag_chain
)

# Invoke with session memory
resp = with_message_history.invoke(
    {"input":  "what is self-Reflection"},
    config={"configurable": {"session_id": "chat1"}}
)['answer']

print(resp)

resp = with_message_history.invoke(
    {"input":  "How to achieve it"},
    config={"configurable": {"session_id": "chat1"}}
)['answer']

print(resp)


Self-reflection is the process of examining and understanding your own thoughts, feelings, and behaviors, and how they impact your life and relationships. It involves taking a step back to analyze and evaluate your experiences, emotions, and actions, and making conscious choices to improve and grow.
To achieve self-reflection, you can practice journaling, meditation, and setting aside time for introspection. Ask yourself questions like "What did I learn today?", "What can I improve on?", and "What am I grateful for?" Regularly taking time to reflect on your thoughts, feelings, and actions can help you gain a deeper understanding of yourself.
